In [0]:
from datetime import datetime
import zoneinfo

# ---------------------------------------------------------
# 1. 設定内容（F03：東北第三工場に切り替え）
# ---------------------------------------------------------
FACTORY_ID = "F03"

jst = zoneinfo.ZoneInfo("Asia/Tokyo")
current_time = datetime.now(jst).strftime("%Y%m%d-%H%M%S")
file_name = f"cf_mom_factory_raw_logs_{FACTORY_ID}_{current_time}.csv"
# ※バケット名やパスは環境に合わせて調整してください
target_path = f"s3://cf-mom-factory-bucket-dev/mom_factory_dev/raw/{file_name}"

# ---------------------------------------------------------
# 2. あえて不整合・エラーを仕込んだテスト用CSVデータ
# ---------------------------------------------------------
csv_content = f"""log_id,vin,timestamp,factory_id,line_id,process_code,car_model_code,process_result_code,cycle_time_sec,defect_code,operator_id,parts_lot_no,recipe_id,4m_changed_flg,sequence_no
F03-L0001,VIN-F03-00001,2026/6/3 10:00,F03,LINE-A,P05,BZ4X-EV,1,60.0,NONE,OP-106,LOT-BAT-101,RCP-01,FALSE,1
,VIN-F03-00002,2026/6/3 10:05,F03,LINE-A,P05,BZ4X-EV,1,55.5,NONE,OP-106,LOT-BAT-101,RCP-01,FALSE,2
F03-L0003,VIN-F03-00003,2026-06-03 10:10:00,F03,LINE-A,P05,BZ4X-EV,1,58.2,NONE,OP-106,LOT-BAT-101,RCP-01,FALSE,3
F03-L0004,VIN-F03-00004,2026/6/3 10:15,F03,LINE-A,P05,BZ4X-EV,1,-45.5,NONE,OP-106,LOT-BAT-101,RCP-01,FALSE,4
F03-L0005,VIN-F03-00005,2026/6/3 10:20,F03,LINE-A,P05,BZ4X-EV,1,NONE,NONE,OP-106,LOT-BAT-101,RCP-01,FALSE,5
F03-L0006,VIN-F03-00006,2026/6/3 10:25,F03,LINE-A,P05,BZ4X-EV,1,62.0,NONE,OP-106,LOT-BAT-101,RCP-01,FALSE,6,HEV,EXTRA_COLUMN_DATA
F03-L0007,VIN-F03-00007,2026-06-03 10:30:00,F03,LINE-A,P05,BZ4X-EV,1,-999.9,NONE,OP-106,LOT-BAT-101,RCP-01,FALSE,7
,VIN-F03-00008,2026/6/3 10:35,F03,LINE-A,P05,BZ4X-EV,1,NONE,NONE,OP-106,LOT-BAT-101,RCP-01,FALSE,8
"""

# ---------------------------------------------------------
# 3. ファイル書き込み実行
# ---------------------------------------------------------
try:
    dbutils.fs.put(target_path, csv_content.strip(), overwrite=True)
    print(f"SUCCESS: {target_path}")
    print("生成されたCSVの各行のエラーシミュレーション内容:")
    print("  1行目 (L0001): 【正常】すべて正常（F03マスターに完全に一致）")
    print("  2行目 (L0002): 【CRITICAL】主キー欠損（log_idが完全に空っぽ）")
    print("  3行目 (L0003): 【ERROR】日付不正（スラッシュではなくハイフン区切り）")
    print("  4行目 (L0004): 【ERROR】異常値（cycle_time_secがマイナス値）")
    print("  5行目 (L0005): 【ERROR】異常値（cycle_time_secが文字列の'NONE'）")
    print("  6行目 (L0006): 【CRITICAL】スキーマ破壊（ヘッダーの15列を超える17列目のデータをあえて注入）")
    print("  7行目 (L0007): 【マルチ】ERROR × 2（日付不正 ＋ サイクルタイム爆発）")
    print("  8行目 (L0008): 【マルチ】CRITICAL ＋ ERROR（主キー欠損 ＋ サイクルタイムNONE）")

except Exception as e:
    print(f"ERROR: {str(e)}")